# Univariate financial ratios and observed defaults
Inspired by Moody's RiskCalc, Section IV (pp. 27–31): rank observations by ratio, form up to 50 groups, and smooth observed default frequency. This is descriptive EDA, not the proprietary RiskCalc transformation or a fitted PD model.

Default labels use the existing 12-month window. By default this uses round 3 training firm-years (2007–2009); the same company can contribute multiple observations. Do not use later test years for feature selection. Ratios are computed after the existing training-only imputation, before VIF and normalization. Imputed values and zero-denominator conventions influence the curves.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config/feature_schema.json').exists())
sys.path.insert(0, str(ROOT / 'src'))
from finance_project.data_processing.preprocess import PreprocessingConfig, _missing, _build_features

ROUND = '03'  # training anchors only; change for another round
N_BINS = 50
SMOOTH_WINDOW = 5  # centered rolling window, weighted by bin counts
round_dir = ROOT / 'data/processed/walk_forward_temporal' / f'round_{ROUND}'
train = pd.read_csv(round_dir / 'train.csv', low_memory=False)
history = pd.read_csv(round_dir / 'history.csv', low_memory=False)
config = PreprocessingConfig(feature_mode='firm_year')
medians = _missing['sector_medians'](train)
features = _build_features(train, history, medians, config)
schema = json.loads((ROOT / 'config/feature_schema.json').read_text())
groups = {k: [x for x in schema['feature_groups'][k] if x in schema['period_numeric_features'] and x not in {'profit', 'rev_operating', 'ebitda', 'COGS', 'prof_operations', 'exp_financing'}] for k in ['leverage_capital_structure', 'liquidity_activity', 'profitability_growth', 'coverage_cash_flow']}
print('Training firm-years:', len(features), '| observed default rate:', f'{features.label.mean():.2%}')
print('Ratio groups:', groups)

## Default-frequency curves
Equal-frequency bins are formed from actual feature values; duplicate quantile edges are merged to avoid splitting identical values. Smoothing is a count-weighted five-bin rolling average, a transparent approximation rather than Moody's proprietary smoothing. Shaded 95% Wilson intervals are for individual bins, not the smoothed curve; they assume independent records and understate uncertainty when firms repeat.

In [ ]:
def bin_feature(frame, feature, n_bins=50, window=5):
    data = frame[[feature, 'label']].copy()
    data[feature] = pd.to_numeric(data[feature], errors='coerce').replace([np.inf, -np.inf], np.nan)
    data = data.dropna()
    if data.empty or data[feature].nunique() < 2:
        return pd.DataFrame()
    data['bin'] = pd.qcut(data[feature], q=n_bins, duplicates='drop')
    out = data.groupby('bin', observed=True).agg(n=(feature, 'size'), defaults=('label', 'sum'), value_min=(feature, 'min'), value_max=(feature, 'max'), value_median=(feature, 'median')).reset_index(drop=True)
    out['default_rate'] = out.defaults / out.n
    out['percentile'] = (out.n.cumsum() - out.n / 2) / out.n.sum()
    out['smooth_rate'] = out.defaults.rolling(window, center=True, min_periods=1).sum() / out.n.rolling(window, center=True, min_periods=1).sum()
    z = 1.96
    denom = 1 + z*z/out.n
    center = (out.default_rate + z*z/(2*out.n))/denom
    half = z*np.sqrt(out.default_rate*(1-out.default_rate)/out.n + z*z/(4*out.n**2))/denom
    out['ci_low'], out['ci_high'] = center-half, center+half
    return out

tables = {}
for group, names in groups.items():
    print('\n', group.upper().replace('_', ' '))
    for name in names:
        if name not in features:
            print('Unavailable:', name); continue
        table = bin_feature(features, name, N_BINS, SMOOTH_WINDOW)
        if table.empty:
            print('No usable variation:', name); continue
        tables[(group, name)] = table
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(table.percentile, table.default_rate, '.', color='steelblue', alpha=.5, label='Observed bin rate')
        ax.plot(table.percentile, table.smooth_rate, color='firebrick', linewidth=2, label='Smoothed rate')
        ax.fill_between(table.percentile, table.ci_low, table.ci_high, color='steelblue', alpha=.15, label='Bin 95% Wilson interval')
        ax.axhline(features.label.mean(), color='gray', linestyle='--', label='Overall training default rate')
        ax.set(title=f'{name}: observed 12-month default rate', xlabel='Feature percentile (low to high)', ylabel='Observed default rate')
        ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.legend(); fig.tight_layout(); plt.show()
        display(table.round(4))

## Binary flags: separate 0 versus 1 comparison
Binary flags have two meaningful states, so quantile curves are inappropriate. Compare counts, defaults and observed rates by state. Flags can reveal data-quality or denominator conditions, but rare states are noisy and a flag may duplicate information already captured by another feature. Associations are not causal; evaluate incremental value and stability separately.

In [ ]:
binary_features = [c for c in schema['period_binary_features'] if c in features]
flag_results = []
for name in binary_features:
    tmp = features[[name, 'label']].dropna()
    for state in (0, 1):
        labels = tmp.loc[tmp[name] == state, 'label']
        flag_results.append({'feature': name, 'state': state, 'records': len(labels), 'defaults': int(labels.sum()), 'default_rate': labels.mean() if len(labels) else np.nan})
flags = pd.DataFrame(flag_results)
display(flags)
rates = flags.pivot(index='feature', columns='state', values='default_rate')
rates.plot.barh(figsize=(10, max(5, len(rates)*.4)), title='Observed default rates by flag state')
plt.gca().xaxis.set_major_formatter(PercentFormatter(1)); plt.xlabel('Observed default rate'); plt.tight_layout(); plt.show()